In [1]:
import time; T0 = time.time()
!pip install -q deepspeed bitsandbytes peft opensmile
!pip uninstall -y -q torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, re, json, glob, shutil, subprocess
!git clone -q https://github.com/zehuiwu/SpeechCueLLM /kaggle/working/SpeechCueLLM
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

In [3]:
hits = [h for h in glob.glob("/kaggle/input/**/processed_iemocap_audio_features_5.csv", recursive=True)
        if "/B3_iemocap/" in h]
print(hits)                      # expect exactly one path
assert len(hits) == 1, f"expected 1 B3 file, got {len(hits)}"
B2 = hits[0]
DST = "/kaggle/working/SpeechCueLLM/speech_features/processed_iemocap_audio_features_5.csv"
shutil.copy(DST, "/kaggle/working/processed_iemocap_RAW_backup.csv")
shutil.copy(B2, DST)

import pandas as pd
d = pd.read_csv(DST); print(d.shape)
print(d['description'].iloc[0])

['/kaggle/input/datasets/kushalag21/iemocap-with-snr/B3_iemocap/processed_iemocap_audio_features_5.csv']
(10039, 25)
Target speech characteristics (estimated SNR 14 dB, low audio reliability): possibly moderate volume with moderate variation, possibly very low pitch with very low variation, possibly high speaking rate.


In [4]:
MODEL_PATH = "/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1"
EPOCHS = 3
DATASET, LR = "iemocap", "3e-4"
RANK, DORA = 16, True          # P2: 8/False | P3: 32/False | P4: 16/True
TAG = f"r{RANK}" + ("_dora" if DORA else "")
assert os.path.exists(MODEL_PATH + "/config.json")

In [5]:
s = open("main.py").read()
s = s.replace(
"        model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
"""        import time, gc, torch as _t
        if int(os.environ.get("LOCAL_RANK", 0)) == 1:
            time.sleep(180)
        from transformers import BitsAndBytesConfig
        _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_t.float16)
        model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, quantization_config=_bnb,
                    torch_dtype=_t.float16, low_cpu_mem_usage=True,
                    device_map={"": int(os.environ.get("LOCAL_RANK", 0))})
        gc.collect()
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(args.gradient_checkpointing))""")
s = s.replace('deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
              'deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
s = s.replace("    return optimizer_grouped_parameters",
              "    return [g for g in optimizer_grouped_parameters if len(g['params']) > 0]")
s = s.replace("    deepspeed_config[\"zero_optimization\"]['mics_shard_size'] = world_size", "    pass")
open("main.py", "w").write(s)

p = "data_utils/deepspeed_config.json"
cfg = json.load(open(p)); cfg.pop("amp", None); cfg["zero_optimization"]["stage"] = 2
json.dump(cfg, open(p, "w"), indent=2)
print("4bit", "BitsAndBytesConfig" in s, "| fp16", 'deepspeed_config["fp16"]["enabled"] = True' in s, "| amp removed", "amp" not in cfg)

4bit True | fp16 True | amp removed True


In [6]:
sh = open("train_and_inference.sh").read()
def setv(name, val):
    global sh; sh = re.sub(rf"^{name}=.*$", f"{name}={val}", sh, count=1, flags=re.M)
setv("MODEL_NAME", "'LLaMA2'"); setv("Experiments_setting", "'lora'"); setv("dataset", f"'{DATASET}'")
setv("audio_description", "'True'"); setv("audio_context", "'True'")
setv("audio_impression", "'False'"); setv("audio_only", "'False'")
setv("SEED", 11); setv("num_train_epochs", EPOCHS); setv("LORA_LR", LR)
setv("accumulations", 4); setv("graphics_card", 2); setv("data_percent", 1.0)
sh = sh.replace("MODEL_PATH='LLaMA2 MODELPATH'", f"MODEL_PATH='{MODEL_PATH}'")
sh = sh.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open("train_and_inference.sh", "w").write(sh)
!grep -n -E "^(audio_|dataset|num_train|data_percent|SEED)" train_and_inference.sh

21:dataset='iemocap'
25:audio_description='True'
26:audio_impression='False'
27:audio_context='True'
28:audio_only='False'
31:SEED=11
32:num_train_epochs=3
55:data_percent=1.0


In [7]:
s = open("main.py").read()
s = s.replace("    bias='none',\n)", f"    bias='none',\n    use_dora={DORA},\n)", 1)
s = s.replace("    model = get_peft_model(model, lora_config)",
              "    model = get_peft_model(model, lora_config)\n    model.print_trainable_parameters()", 1)
open("main.py", "w").write(s)

sh = open("train_and_inference.sh").read()
sh = sh.replace("--gradient_checkpointing\n    fi",
                f"--gradient_checkpointing \\\n        --lora_dim {RANK} --lora_alpha {RANK}\n    fi", 1)
open("train_and_inference.sh", "w").write(sh)
print(f"use_dora={DORA}" in s, f"--lora_dim {RANK}" in sh)   # expect True True

True True


In [8]:
LOG = f"/kaggle/working/run_log_{TAG}_iemocap_7B.txt"
proc = subprocess.Popen(f"PYTHONUNBUFFERED=1 bash train_and_inference.sh > {LOG} 2>&1", shell=True)
print("started, pid", proc.pid)

started, pid 140


In [9]:
DEADLINE_H = 11.0

def status():
    lines = open(LOG, errors="ignore").read().replace("\r", "\n").split("\n")
    steps = [l for l in lines if "Running Loss" in l]
    last = steps[-1].strip()[-140:] if steps else "(loading model / not started)"
    f1 = [l for l in lines if l.startswith(("current_f1", "best_f1", "history"))][-3:]
    err = [l for l in lines if "Traceback" in l or "OutOfMemory" in l or "Killed" in l][-2:]
    gpu = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    ram = subprocess.run("free -g | awk 'NR==2{print $3\"/\"$2\" GB\"}'", shell=True, capture_output=True, text=True).stdout.strip()
    print(f"[{time.strftime('%H:%M:%S')} +{(time.time()-T0)/3600:.1f}h] {last}\n  GPU: {gpu} | RAM: {ram}")
    for l in f1 + err: print("  ", l.strip()[:200])

while proc.poll() is None:
    status(); time.sleep(300)
    if time.time() - T0 > DEADLINE_H * 3600:
        print("DEADLINE reached, stopping training"); subprocess.run("pkill -f main.py", shell=True); break
status()
subprocess.run("pkill -f main.py; pkill -f deepspeed", shell=True)
print("FINISHED, exit code", proc.poll())

[19:58:44 +0.0h] (loading model / not started)
  GPU: 0 MiB, 0 % | 0 MiB, 0 % | RAM: 1/31 GB
[20:03:44 +0.1h] (loading model / not started)
  GPU: 6425 MiB, 0 % | 3 MiB, 0 % | RAM: 3/31 GB
[20:08:44 +0.2h] Epochs 0/3. Running Loss:    2.0156:   5%|▌         | 131/2582 [04:40<1:24:55,  2.08s/it]
  GPU: 5447 MiB, 93 % | 5361 MiB, 100 % | RAM: 5/31 GB
[20:13:44 +0.3h] Epochs 0/3. Running Loss:    0.6244:  10%|█         | 269/2582 [09:41<1:21:30,  2.11s/it]
  GPU: 5533 MiB, 100 % | 5361 MiB, 100 % | RAM: 5/31 GB
[20:18:44 +0.3h] Epochs 0/3. Running Loss:    1.5988:  16%|█▌        | 407/2582 [14:41<1:18:07,  2.16s/it]
  GPU: 5533 MiB, 100 % | 5361 MiB, 100 % | RAM: 5/31 GB
[20:23:44 +0.4h] Epochs 0/3. Running Loss:    0.8096:  21%|██        | 543/2582 [19:42<1:22:16,  2.42s/it]
  GPU: 5533 MiB, 100 % | 5361 MiB, 100 % | RAM: 5/31 GB
[20:28:44 +0.5h] Epochs 0/3. Running Loss:    0.6478:  26%|██▋       | 680/2582 [24:41<1:09:36,  2.20s/it]
  GPU: 5533 MiB, 100 % | 5361 MiB, 100 % | RAM: 5/31 